# Breast Cancer Classification — Reproducible Analysis

**MSc Data Science | Applied Machine Learning and Deep Learning (MS4S16)**

This notebook revisits the original assignment with a leakage-safe preprocessing pipeline. It uses the supplied, modified Wisconsin Breast Cancer Dataset. It is a **teaching example, not a medical diagnostic system**.

In [1]:
from pathlib import Path
import os, sys
from IPython.display import display
root = Path.cwd().resolve()
if not (root / "scripts" / "train_and_evaluate.py").exists():
    raise FileNotFoundError("Open Jupyter from the repository root directory")
sys.path.insert(0, str(root))
from scripts.train_and_evaluate import load_and_split, train_models, save_plots
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt
import pandas as pd

## 1. Data inspection

The CSV includes 30 numeric features, a diagnosis label (B/M), and a numeric ID excluded from modelling. Rows without diagnosis and duplicate records are removed **before** the stratified split.

In [2]:
split, data_summary = load_and_split()
print(data_summary)
X_train, X_test, y_train, y_test = split
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

{'raw_rows': 571, 'clean_rows': 566, 'removed_missing_target_and_duplicate_rows': 5, 'features': 30, 'missing_numeric_cells_after_sentinel_conversion': 261, 'class_counts': {0: 355, 1: 211}}
Training rows: 452 | Test rows: 114


## 2. Model training without data leakage

Each model uses a scikit-learn `Pipeline`, so missing-value imputation and scaling are fitted **only on training data**. Logistic Regression uses standard scaling; Random Forest does not require scaling. Both use an 80/20 stratified split and seed 42.

In [3]:
comparison, outputs = train_models(split)
display(comparison.round(4))

,model,accuracy,precision_malignant,recall_malignant,f1_malignant,roc_auc,false_negatives,false_positives
0,Logistic Regression,0.9561,0.9302,0.9524,0.9412,0.9838,2,3
1,Random Forest,0.9737,1.0000,0.9286,0.9630,0.9964,3,0


## 3. Error analysis

For this classification demonstration, malignant observations are the positive class (`1`). A **false negative** is a malignant sample predicted benign. Inspect false negatives and recall alongside overall accuracy.

In [4]:
for name, values in outputs.items():
    print(name)
    print("Confusion matrix [[TN, FP], [FN, TP]]:", values["confusion_matrix"])
    print(values["classification_report"])

Logistic Regression
Confusion matrix [[TN, FP], [FN, TP]]: [[69, 3], [2, 40]]
              precision    recall  f1-score   support

      Benign       0.97      0.96      0.97        72
   Malignant       0.93      0.95      0.94        42

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

Random Forest
Confusion matrix [[TN, FP], [FN, TP]]: [[72, 0], [3, 39]]
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



## 4. ROC–AUC comparison

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, values in outputs.items():
    RocCurveDisplay.from_predictions(y_test, values["proba"], name=name, ax=ax)
ax.plot([0,1], [0,1], linestyle="--", color="gray")
ax.set_title("Held-out ROC curves")
plt.show()

/tmp/ipykernel_1439/326090025.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Results and limitations

- The **original university notebook** used mean imputation before train/test splitting, creating potential information leakage; its recorded accuracy is **95.61%** (Logistic Regression) and **98.25%** (Random Forest); those results also did not address `-999` sentinel values.
- This notebook reports a **recomputed** evaluation, removing duplicate rows and fitting preprocessing on training data only. Results can therefore differ from the assessment outputs.
- There is one random stratified hold-out split, not repeated cross-validation or external clinical validation. Predictions must **not** be used for medical decisions.